# Восстановление коэффициентов

[Задача на CodeRun](https://coderun.yandex.ru/problem/coefficients-restoration). Даны 1000 точек $(x, y)$:

$$y = (a\sin x + b\ln x)^2 + c\,x^2 + \text{шум}.$$

- $x$, $y$ — данные из файла, они известны;
- $a$, $b$, $c > 0$ — неизвестные коэффициенты, их нужно найти с точностью $10^{-2}$;
- шум — отклонения коэффициентов не больше $0{,}001$. Это в 10 раз меньше нужной точности, поэтому шум просто не учитываем.

**Как решаем.** Берём модель $\hat y = (a\sin x + b\ln x)^2 + c\,x^2$ и подбираем $a$, $b$, $c$ так, чтобы предсказания $\hat y_i$ были как можно ближе к $y_i$. Для этого считаем ошибку, находим её производные по $a$, $b$, $c$ и запускаем градиентный спуск.

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np

URL = "https://assets.contest.yandex.net/testsys/statement-file?hash=eyJhbGciOiJkaXIiLCJlbmMiOiJBMjU2R0NNIn0..S1G3epXKnK0cjv51.l6h0gXNaY9YG2YcFEE8q8DwFzfhZY2OQmoOu-Px4yuNzMK2FyfcFJhosWW9cZ5XUMGOUA0wVdiMZ0U3D5NGyk-OOl7LB37A.7Q6iR5TS6Y7SPME_yxW64w"
if not Path("data.csv").exists():
    urllib.request.urlretrieve(URL, "data.csv")

x, y = np.loadtxt("data.csv", delimiter=",", unpack=True)
print(len(x), "точек")

Если скачать не получилось, возьмите `data.csv` со страницы задачи и положите рядом с ноутбуком.

## 1. Производные

Ошибка — MSE:

$$L = \frac1n\sum_i \mathrm{err}_i^2, \qquad \mathrm{err}_i = (a\sin x_i + b\ln x_i)^2 + c\,x_i^2 - y_i.$$

- $i$ — номер точки, $n = 1000$ — число точек;
- $\mathrm{err}_i = \hat y_i - y_i$ — на сколько модель ошиблась в точке $i$;
- $L$ — средний квадрат ошибки. Он зависит только от $a$, $b$, $c$: данные $x_i$, $y_i$ — просто числа.

Найдите на бумаге $\dfrac{\partial L}{\partial a}$, $\dfrac{\partial L}{\partial b}$, $\dfrac{\partial L}{\partial c}$.

<details>
<summary><b>💡 Подсказка</b></summary>

Цепное правило, причём дважды: снаружи квадрат $\mathrm{err}^2$, а внутри $\mathrm{err}$ ещё один квадрат $(a\sin x + b\ln x)^2$.

</details>

<details>
<summary><b>✅ Решение по шагам</b></summary>

**Шаг 1. Производная суммы — сумма производных.** Достаточно продифференцировать один член $\mathrm{err}_i^2$. Снаружи квадрат:

$$\frac{\partial}{\partial a}\mathrm{err}_i^2 = 2\,\mathrm{err}_i\cdot\frac{\partial\,\mathrm{err}_i}{\partial a}.$$

**Шаг 2. Производная $\mathrm{err}_i$ по $a$.** Обозначим $s_i = a\sin x_i + b\ln x_i$, тогда $\mathrm{err}_i = s_i^2 + c\,x_i^2 - y_i$.

- $c\,x_i^2$ и $y_i$ от $a$ не зависят: их производная $0$.
- $s_i^2$ — снова квадрат: $\dfrac{\partial}{\partial a}s_i^2 = 2 s_i\cdot\dfrac{\partial s_i}{\partial a}$.
- $\dfrac{\partial s_i}{\partial a} = \sin x_i$: $b\ln x_i$ здесь просто число.

Значит, $\dfrac{\partial\,\mathrm{err}_i}{\partial a} = 2 s_i\sin x_i$.

**Шаг 3. Собираем:**

$$\frac{\partial L}{\partial a} = \frac1n\sum_i 2\,\mathrm{err}_i\cdot 2 s_i\sin x_i = \frac4n\sum_i \mathrm{err}_i\, s_i\sin x_i.$$

**По $b$** всё так же, только $\dfrac{\partial s_i}{\partial b} = \ln x_i$:

$$\frac{\partial L}{\partial b} = \frac4n\sum_i \mathrm{err}_i\, s_i\ln x_i.$$

**По $c$** проще: $\dfrac{\partial\,\mathrm{err}_i}{\partial c} = x_i^2$, поэтому

$$\frac{\partial L}{\partial c} = \frac2n\sum_i \mathrm{err}_i\, x_i^2.$$

В коде $\frac1n\sum_i$ — это `np.mean`.

</details>

## 2. Градиентный спуск

Каждый шаг сдвигает коэффициенты против градиента: $a \leftarrow a - \eta\,\dfrac{\partial L}{\partial a}$, и так же для $b$ и $c$. Здесь $\eta$ — шаг (`lr`).

В коде `s` — это $a\sin x + b\ln x$, `err` — массив ошибок по всем точкам. Допишите градиенты по формулам из пункта 1 и запустите.

In [ ]:
a, b, c = 1.0, 1.0, 1.0
lr = 1e-4

for step in range(5000):
    s = a * np.sin(x) + b * np.log(x)
    err = s**2 + c * x**2 - y

    grad_a = ...  # ваш код
    grad_b = ...
    grad_c = ...

    a -= lr * grad_a
    b -= lr * grad_b
    c -= lr * grad_c

    if step % 1000 == 0:
        print(step, np.mean(err**2))

print(f"{a:.2f} {b:.2f} {c:.2f}")

<details>
<summary><b>✅ Решение</b></summary>

```python
grad_a = 4 * np.mean(err * s * np.sin(x))
grad_b = 4 * np.mean(err * s * np.log(x))
grad_c = 2 * np.mean(err * x**2)
```

</details>

<details>
<summary><b>🤔 Ошибка растёт или получается nan?</b></summary>

Скорее всего, шаг слишком большой. $x$ доходит до 10, значит $x^2$ — до 100, и градиент по $c$ огромный: уже при `lr = 1e-3` спуск разлетается. Верните `lr = 1e-4`.

Если шаг `1e-4`, а ошибка всё равно растёт, проверьте формулы градиентов: знаки и множители 4 и 2.

</details>


Последняя строка — ответ для CodeRun.